# C2 Requests Handling — Transport Wait Analysis

Computes per-patient transport waiting times from the patient log
(`WAITING_TRANSPORT:*` → `CLOCK_PAUSED:transport`) at both CCP and R1,
then cross-references the C2 dispatch logs to identify **T1s that asked
for an ambulance when none was available**.

Key questions:
- How many T1s were denied an ambulance (no idle ambulances at dispatch time)?
- How long did those starved T1s wait before eventually being picked up?
- What is the distribution of transport waiting times per triage?
- How do CCP→R1 waits compare to R1→R2 waits?

In [ ]:
import sys
from pathlib import Path
_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
import paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

run_folder = paths.SINGLE_RUN_OUTPUT / FOLDER_NAME

pl = pd.read_excel(run_folder / "patient_log.xlsx")
print(f"Loaded {len(pl)} log rows, {pl['patient_id'].nunique()} patients")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# COMPUTE PER-PATIENT TRANSPORT WAITS
# ═══════════════════════════════════════════════════════════════════════════
# For each patient, find every WAITING_TRANSPORT:* event and measure the
# time until the next CLOCK_PAUSED:transport (= ambulance pickup).
# Patients who died while waiting have no pickup — they are flagged.

LOCATION_MAP = {
    "WAITING_TRANSPORT:ccp":   "CCP",
    "WAITING_TRANSPORT:role1": "R1",
}

records = []
for pid, grp in pl.groupby("patient_id"):
    rows = grp.sort_values("time")
    actions = rows["action"].tolist()
    times   = rows["time"].tolist()
    triage  = rows["triage"].iloc[0]
    t_short = triage.split("_")[0] if "_" in str(triage) else str(triage)
    n = len(actions)

    for i, (t, a) in enumerate(zip(times, actions)):
        if a not in LOCATION_MAP:
            continue
        location = LOCATION_MAP[a]
        # find next CLOCK_PAUSED:transport or DIED
        pickup_time = None
        died_waiting = False
        for j in range(i + 1, n):
            if actions[j] == "CLOCK_PAUSED:transport":
                pickup_time = times[j]
                break
            if actions[j] == "DIED":
                died_waiting = True
                break
        if pickup_time is not None:
            records.append({
                "patient_id": pid,
                "triage": t_short,
                "location": location,
                "t_request": t,
                "t_pickup": pickup_time,
                "wait_min": pickup_time - t,
                "outcome": "picked_up",
            })
        elif died_waiting:
            records.append({
                "patient_id": pid,
                "triage": t_short,
                "location": location,
                "t_request": t,
                "t_pickup": np.nan,
                "wait_min": np.nan,
                "outcome": "died_waiting",
            })

waits = pd.DataFrame(records)
picked_up = waits[waits["outcome"] == "picked_up"]
died_w    = waits[waits["outcome"] == "died_waiting"]

print(f"Transport waits: {len(picked_up)} picked up, {len(died_w)} died waiting")
print()
print(picked_up.groupby(["location", "triage"])["wait_min"]
      .agg(["count", "mean", "median", "max"]).round(1))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# T1 STARVED — C2 tried to dispatch but had NO idle ambulances
# ═══════════════════════════════════════════════════════════════════════════
# Load C2 dispatch logs and find T1 patient_ids that were assigned to an
# ambulance.  A T1 is "starved" if no dispatch row near its request time
# contains its id in assignments — meaning C2 had no idle ambulance when
# the T1 first triggered a wake-up.
#
# Because C2 only logs when idle_ambulances > 0 (the dispatch loop doesn't
# run otherwise), a starved T1 will first appear in a LATER log row.
# We detect this by comparing t_request (patient log) with the earliest
# C2 assignment time for that patient.

import json

C2_MAP = {
    "CCP": "c2_ccp_r1",
    "R1":  "c2_r1_r2",
}

def _load_c2(name):
    path = run_folder / f"{name}.xlsx"
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_excel(path)
    for col in ["transport_requests", "assignments"]:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    return df

def _build_assignment_index(c2_df):
    """Map patient_id → earliest assignment time from C2 log."""
    idx = {}
    for _, row in c2_df.iterrows():
        assigns = row.get("assignments", {})
        if not isinstance(assigns, dict):
            continue
        t = row["time"]
        for pids in assigns.values():
            for pid in pids:
                if pid not in idx or t < idx[pid]:
                    idx[pid] = t
    return idx

for loc in ["CCP", "R1"]:
    c2_name = C2_MAP[loc]
    c2_df = _load_c2(c2_name)
    if c2_df.empty:
        print(f"{loc}: {c2_name}.xlsx not found or empty")
        continue

    assign_times = _build_assignment_index(c2_df)

    t1 = picked_up[(picked_up["triage"] == "T1") & (picked_up["location"] == loc)].copy()
    if t1.empty:
        print(f"{loc}: no T1 transport waits")
        continue

    t1["t_assigned"] = t1["patient_id"].map(assign_times)
    t1["dispatch_delay"] = t1["t_assigned"] - t1["t_request"]

    # "Served immediately" = assigned in the same dispatch episode as the
    # request (dispatch_delay ≈ 0, allowing a small tolerance for float
    # arithmetic inside the simulation clock).
    TOLERANCE = 0.5  # minutes
    immediate = t1[t1["dispatch_delay"] <= TOLERANCE]
    starved   = t1[t1["dispatch_delay"] > TOLERANCE]
    unknown   = t1[t1["t_assigned"].isna()]

    print(f"{loc}:  T1 total: {len(t1)}  |  "
          f"served immediately: {len(immediate)}  |  "
          f"starved (no ambulance on first attempt): {len(starved)}  "
          f"({100*len(starved)/len(t1):.1f}%)")
    if not unknown.empty:
        print(f"  ⚠ {len(unknown)} T1s not found in C2 assignments")
    if not starved.empty:
        print(f"  Starved T1 total wait:  mean={starved['wait_min'].mean():.1f}  "
              f"median={starved['wait_min'].median():.1f}  "
              f"max={starved['wait_min'].max():.1f} min")
        print(f"  Starved T1 dispatch delay:  mean={starved['dispatch_delay'].mean():.1f}  "
              f"median={starved['dispatch_delay'].median():.1f}  "
              f"max={starved['dispatch_delay'].max():.1f} min")
    print()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HISTOGRAM — T1 transport waits: served immediately vs starved
# ═══════════════════════════════════════════════════════════════════════════

TRIAGE_COLORS = {"T1": "#e74c3c", "T2": "#f39c12", "T3": "#3498db"}
TOLERANCE = 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, loc in zip(axes, ["CCP", "R1"]):
    c2_df = _load_c2(C2_MAP[loc])
    if c2_df.empty:
        ax.set_title(f"{loc}: no C2 log")
        continue
    assign_times = _build_assignment_index(c2_df)

    t1 = picked_up[(picked_up["triage"] == "T1") & (picked_up["location"] == loc)].copy()
    if t1.empty:
        ax.set_title(f"{loc}: no T1 waits")
        continue

    t1["t_assigned"] = t1["patient_id"].map(assign_times)
    t1["dispatch_delay"] = t1["t_assigned"] - t1["t_request"]

    immediate = t1[t1["dispatch_delay"] <= TOLERANCE]
    starved   = t1[t1["dispatch_delay"] > TOLERANCE]

    bins = np.linspace(0, t1["wait_min"].max() * 1.05, 25)
    if not immediate.empty:
        ax.hist(immediate["wait_min"], bins=bins, edgecolor="black",
                color="#2ecc71", alpha=0.7, label=f"immediate (n={len(immediate)})")
    if not starved.empty:
        ax.hist(starved["wait_min"], bins=bins, edgecolor="black",
                color="#e74c3c", alpha=0.7, label=f"starved (n={len(starved)})")
    ax.set_xlabel("Total wait time (min)")
    ax.set_ylabel("T1 patients")
    ax.set_title(f"T1 transport wait @ {loc}")
    ax.legend()

plt.suptitle(f"T1 transport waits — immediate vs starved — {FOLDER_NAME}",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DISTRIBUTION OF ALL TRIAGE TRANSPORT WAITS — CCP and R1 side by side
# ═══════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, loc in zip(axes, ["CCP", "R1"]):
    sub = picked_up[picked_up["location"] == loc]
    for triage in ["T1", "T2", "T3"]:
        t_data = sub[sub["triage"] == triage]["wait_min"]
        if t_data.empty:
            continue
        ax.hist(t_data, bins=25, alpha=0.6, edgecolor="black", linewidth=0.5,
                color=TRIAGE_COLORS[triage], label=f"{triage} (n={len(t_data)})")
    ax.set_xlabel("Wait time (min)")
    ax.set_ylabel("Patients")
    ax.set_title(f"Transport wait @ {loc}")
    ax.legend()
plt.suptitle(f"Transport wait distribution — {FOLDER_NAME}", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BOX PLOT — transport wait by triage × location
# ═══════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, loc in zip(axes, ["CCP", "R1"]):
    sub = picked_up[picked_up["location"] == loc]
    data = [sub[sub["triage"] == t]["wait_min"].values for t in ["T1", "T2", "T3"]]
    bp = ax.boxplot(data, labels=["T1", "T2", "T3"], patch_artist=True,
                    showfliers=True, flierprops={"markersize": 3})
    for patch, t in zip(bp["boxes"], ["T1", "T2", "T3"]):
        patch.set_facecolor(TRIAGE_COLORS[t])
        patch.set_alpha(0.7)
    ax.set_ylabel("Wait time (min)")
    ax.set_title(f"Transport wait @ {loc}")
plt.suptitle(f"Transport wait by triage — {FOLDER_NAME}", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DIED WHILE WAITING FOR TRANSPORT
# ═══════════════════════════════════════════════════════════════════════════

if died_w.empty:
    print("No patients died while waiting for transport.")
else:
    print("Patients who died waiting for transport:")
    print(died_w.groupby(["location", "triage"])["patient_id"].count()
          .rename("n_died").to_frame())